# Location Selection with E-NAUTILUS: Part 1 (with RPM)
_Generation of reference points_

In [1]:
from desdeo.problem import Constant, Problem, Objective, VariableTypeEnum, Constraint, TensorConstant, TensorVariable, ConstraintTypeEnum
import numpy as np
import pandas as pd
from slugify import slugify
# These are to just suppress warnings in the outputs of the example
import warnings
import pickle

warnings.filterwarnings("ignore")

data_home = "../../data"

## Model inputs


In [2]:

# Mininum expected attendance to be worth visiting 
min_att = 4                                                            # Reviewed

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.90

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 9.0                                                          # 8-10 according to MR
# How long the event is (hours)

# TODO Add the drive time automatically (2 for event + drive time)
hours_per_event = 4                                                    # 
driver_salary_per_hour = 20                                            # Reviewed
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour    # Reviewed

# Food desert threshold (minutes)
# 10 miles according to the USDA (Mike to send email)
# 15 minute travel time according to 
# https://www.goodrx.com/healthcare-access/research/healthcare-deserts-80-percent-of-country-lacks-adequate-healthcare-access
# https://www.goodrx.com/hcp/clinical-resources/prescriptions-hcp/pharmacy-deserts
raw_food_desert_threshold = 15                                         # Reviewed
                                                                       # Other costs?


max_events_raw = 16



## Helper functions

In [3]:
def no_nan(val):
    if pd.isna(val):
        return ""
    else:
        return str(val)
    

## Load and process Constants

In [4]:


home = "Ada"
# Read the adjacency matrix (distance in miles)
adjDist = pd.read_csv(f"{data_home}/adjacencyMatrixDist.csv", index_col=0)
dist2home = adjDist.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"
display("Distance to home base")
display(dist2home)

# Read the adjacency matrix (travel time in minutes)
adjTTime = pd.read_csv(f"{data_home}/adjacencyMatrixTravelTime.csv", index_col=0)
display("Travel time adjacency matrix")
display(adjTTime)

# Read the cities 
cities = pd.read_csv(f"{data_home}/cities.csv")

# Data integrity check
cities_cities_df = set(cities.loc[:,"city"])
cities_adjDist_df = set(adjDist.index)
if cities_cities_df != cities_adjDist_df: 
    raise ValueError(f"The city table and the adjacency matrix have differing cities.\nCity table: {cities_cities_df}\nAdjacency matrix{cities_adjDist_df}")


'Distance to home base'

,dist2home
city,
Ada,0.00
Alger,5.24
Bluffton,11.44
Cairo,19.53
Caledonia,63.65
Carey,40.15
Columbus Grove,22.75
Continental,44.40
Cridersville,24.66


'Travel time adjacency matrix'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Marysville,Plain City,Richwood,Milford Center,Findlay,Fostoria,McComb,Arlington,Rawson,Arcadia
city,,,,,,,,,,,,,,,,,,,,,
Ada,0,10,19,25,71,46,32,66,40,38,...,69,83,65,71,34,60,43,24,27,47
Alger,10,0,27,33,79,55,40,74,39,46,...,60,74,65,62,42,68,51,32,35,55
Bluffton,18,27,0,20,74,38,21,53,34,34,...,86,100,80,88,21,48,28,20,15,34
Cairo,25,33,21,0,80,54,11,45,31,18,...,88,102,86,90,38,64,47,33,31,51
Caledonia,71,79,73,79,0,47,87,120,98,92,...,57,60,38,63,64,71,79,62,80,73
Carey,47,55,37,54,48,0,57,81,67,67,...,83,86,63,89,23,26,39,29,40,30
Columbus Grove,33,41,21,11,88,58,0,34,40,27,...,97,111,94,99,40,66,37,40,31,53
Continental,66,74,53,45,121,82,34,0,70,36,...,127,141,127,129,62,83,46,72,58,74
Cridersville,39,39,34,31,99,67,40,70,0,42,...,79,93,97,81,51,77,60,50,44,64


### Site table 

In [5]:
sites_only = pd.read_csv(f"{data_home}/proposedSitesProcessed.csv", index_col=0)

# Data integrity check
cities_cities_df = set(cities.loc[:,"city"])
cities_adjDist_df = set(sites_only.loc[:,"city"])
if len(cities_adjDist_df - cities_cities_df) != 0: 
    print(cities_adjDist_df - cities_cities_df)
    raise ValueError(f"Some cities in the site table don't appear in the city table.\nCity table: {cities_cities_df}\nSite table{cities_adjDist_df}")

# Create event table
sites = pd.merge(sites_only,cities,  on="city")
display(sites)
sites.loc[:, "site_id"] = sites.apply(lambda row: slugify(f'{row["city"]} {row["siteName"]}'), axis=1)
sites.loc[:, "site_pretty"] = sites.apply(lambda row: f'{row["siteName"]}', axis=1)

# Add the distance to home for each site 
sites = pd.merge(sites, dist2home, on="city")

display(sites)

,siteName,siteType,city,minAtt,medAtt,maxAtt,lat,long,pop,svi
0,Alger Dinner,charitable meal,Alger,1.0,3.0,5.0,40.709722,-83.844167,837,0.594200
1,Dunkirk Dinner,charitable meal,Dunkirk,2.0,3.0,7.0,40.788056,-83.642778,774,0.547500
2,Lima ODB,charitable meal,Lima,2.0,3.0,4.0,40.746389,-84.123333,35579,0.923333
3,Saint Marks UMC,charitable meal,Lima,2.0,3.0,7.0,40.746389,-84.123333,35579,0.923333
4,Grace Clinics,clinic,Marion,0.0,3.0,10.0,40.620000,-83.126389,35999,0.932500
...,...,...,...,...,...,...,...,...,...,...
92,Union County Personal Needs Pantry,food pantry,Marysville,0.0,3.0,10.0,40.242220,-83.373610,25571,0.459200
93,Plain City Food Pantry,food pantry,Plain City,0.0,3.0,10.0,40.107780,-83.263890,4065,0.258300
94,REAP Food Pantry,food pantry,Richwood,0.0,3.0,10.0,40.427500,-83.295000,2222,0.426700
95,North Union Personal Needs Pantry,food pantry,Richwood,0.0,3.0,10.0,40.427500,-83.295000,2222,0.426700


,siteName,siteType,city,minAtt,medAtt,maxAtt,lat,long,pop,svi,site_id,site_pretty,dist2home
0,Alger Dinner,charitable meal,Alger,1.0,3.0,5.0,40.709722,-83.844167,837,0.594200,alger-alger-dinner,Alger Dinner,5.24
1,Dunkirk Dinner,charitable meal,Dunkirk,2.0,3.0,7.0,40.788056,-83.642778,774,0.547500,dunkirk-dunkirk-dinner,Dunkirk Dinner,11.19
2,Lima ODB,charitable meal,Lima,2.0,3.0,4.0,40.746389,-84.123333,35579,0.923333,lima-lima-odb,Lima ODB,17.48
3,Saint Marks UMC,charitable meal,Lima,2.0,3.0,7.0,40.746389,-84.123333,35579,0.923333,lima-saint-marks-umc,Saint Marks UMC,17.48
4,Grace Clinics,clinic,Marion,0.0,3.0,10.0,40.620000,-83.126389,35999,0.932500,marion-grace-clinics,Grace Clinics,56.11
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,Union County Personal Needs Pantry,food pantry,Marysville,0.0,3.0,10.0,40.242220,-83.373610,25571,0.459200,marysville-union-county-personal-needs-pantry,Union County Personal Needs Pantry,55.15
93,Plain City Food Pantry,food pantry,Plain City,0.0,3.0,10.0,40.107780,-83.263890,4065,0.258300,plain-city-plain-city-food-pantry,Plain City Food Pantry,69.08
94,REAP Food Pantry,food pantry,Richwood,0.0,3.0,10.0,40.427500,-83.295000,2222,0.426700,richwood-reap-food-pantry,REAP Food Pantry,42.23
95,North Union Personal Needs Pantry,food pantry,Richwood,0.0,3.0,10.0,40.427500,-83.295000,2222,0.426700,richwood-north-union-personal-needs-pantry,North Union Personal Needs Pantry,42.23


### Close cities for events

In [6]:

close_cities = adjTTime < raw_food_desert_threshold
cities_adj2sites = sites.loc[:,["city", "site_id"]].merge(close_cities, left_on="city", right_index=True)
display(cities_adj2sites)
cities_adj2sites = (cities_adj2sites.iloc[:,2:].values).astype(int)
s_adj_raw = cities_adj2sites.T
display(cities_adj2sites.T.shape)
e_adj_raw_list = s_adj_raw.tolist()

,city,site_id,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,...,Marysville,Plain City,Richwood,Milford Center,Findlay,Fostoria,McComb,Arlington,Rawson,Arcadia
0,Alger,alger-alger-dinner,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Dunkirk,dunkirk-dunkirk-dinner,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,True,False,False
2,Lima,lima-lima-odb,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,Lima,lima-saint-marks-umc,False,False,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Marion,marion-grace-clinics,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,Marysville,marysville-union-county-personal-needs-pantry,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,False,False,False,False
93,Plain City,plain-city-plain-city-food-pantry,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
94,Richwood,richwood-reap-food-pantry,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False
95,Richwood,richwood-north-union-personal-needs-pantry,False,False,False,False,False,False,False,False,...,False,False,True,False,False,False,False,False,False,False


(53, 97)

In [7]:
site_dist2home = sites.loc[:,["dist2home"]].T.values
site_dist2home_list = site_dist2home.tolist()

## Constants

In [8]:
site_count = sites.shape[0]

# Expected attenance 
raw_attendance = sites.loc[:,["medAtt"]] 
raw_under_attendance = (raw_attendance < min_att).astype(int)

# Site SVI
raw_svi = sites.loc[:,["svi"]]

svi = TensorConstant(name="Site SVI",
                     symbol="svi", 
                     shape=raw_svi.T.shape,
                     values=raw_svi.T.values.tolist())
display(svi)

# How many people to expect to attend each site
ea = TensorConstant(name="Expected attendance",
                                 symbol="ea", 
                                 shape=raw_attendance.T.shape,
                                 values=raw_attendance.T.values.tolist())
display(ea)

# How many sites are expected to be under attended? 
eua = TensorConstant(name="Expected under-attendance",
                    symbol="eua",
                    shape=raw_under_attendance.T.shape,
                    values=raw_under_attendance.T.values.tolist())

display(eua)

# Distance to home 
sd2h = TensorConstant(name="The distance to drive from home to a site",
                    symbol="sd2h",                        
                    shape=site_dist2home.shape,
                    values=site_dist2home_list)
display(sd2h)

#                                                                               TO REVIEW
mpg = Constant(name="Miles per gallon (miles/gallon)", 
               symbol="mpg", 
               value=raw_mpg)

#                                                                               TO REVIEW
display(mpg)
dpg = Constant(name="Dollars per gallon ($/gallon)", 
               symbol="dpg", 
               value=raw_dollars_per_gallon)

display(dpg)
dcpt = Constant(name="Driver cost per trip ($)", 
                                symbol="dcpt", 
                                value=raw_driver_cost_per_trip)

mpg_inv_times_dpg = Constant(name="(1/(miles per gallon)) x dollars per gallon", 
                                symbol="mpg_inv_times_dpg", 
                                value=(1/raw_mpg)*raw_dollars_per_gallon)


display(dcpt)

# Site adjacency to cities
s_adj = TensorConstant(name="What cities are adjacent to a given site",
                      symbol="s_adj",                      
                      shape=s_adj_raw.shape,
                      values=e_adj_raw_list)
display(s_adj)

city_pops = TensorConstant(name="City populations", 
                           symbol="cpop", 
                           shape=[cities.shape[0],1],
                           values=cities.loc[:,["pop"]].values.tolist()
                           )

display(city_pops)

total_pop = Constant(name="Total population of interest", 
                     symbol="tpop", 
                     value=float(sum(cities.loc[:,"pop"])))

display(total_pop)


max_events = Constant(name="Maximum number of events per month", 
                     symbol="max_events", 
                     value=max_events_raw)


TensorConstant(name='Site SVI', symbol='svi', shape=[1, 97], values=['List', ['List', 0.5942, 0.5475, 0.923333333, 0.923333333, 0.9325, 0.5942, 0.923333333, 0.9183, 0.3217, 0.4508, 0.923333333, 0.39, 0.5942, 0.19, 0.3275, 0.4142, 0.3508, 0.3992, 0.5625, 0.565, 0.1958, 0.1, 0.0675, 0.7117, 0.66, 0.3217, 0.0625, 0.1058, 0.4375, 0.6767, 0.5475, 0.16625, 0.9183, 0.1633, 0.923333333, 0.9325, 0.2633, 0.3483, 0.095, 0.2467, 0.0967, 0.5517, 0.195, 0.6392, 0.4725, 0.3467, 0.923333333, 0.9183, 0.3992, 0.5625, 0.4375, 0.6392, 0.9183, 0.9183, 0.4375, 0.923333333, 0.923333333, 0.4375, 0.6767, 0.4375, 0.4725, 0.4725, 0.4725, 0.39, 0.9183, 0.9183, 0.9183, 0.9183, 0.565, 0.923333333, 0.923333333, 0.923333333, 0.923333333, 0.5517, 0.923333333, 0.923333333, 0.4142, 0.6392, 0.923333333, 0.9183, 0.9183, 0.923333333, 0.5625, 0.9325, 0.9325, 0.9325, 0.8125, 0.8125, 0.8125, 0.8125, 0.8125, 0.4592, 0.4592, 0.2583, 0.4267, 0.4267, 0.7533]])

TensorConstant(name='Expected attendance', symbol='ea', shape=[1, 97], values=['List', ['List', 3.0, 3.0, 3.0, 3.0, 3.0, 4.0, 6.0, 2.0, 3.0, 7.0, 3.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 6.0, 3.0, 3.0, 3.0, 7.0, 3.0, 3.0, 3.0, 4.0, 1.0, 1.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]])

TensorConstant(name='Expected under-attendance', symbol='eua', shape=[1, 97], values=['List', ['List', 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])

TensorConstant(name='The distance to drive from home to a site', symbol='sd2h', shape=[1, 97], values=['List', ['List', 5.24, 11.19, 17.48, 17.48, 56.11, 5.24, 17.48, 15.24, 29.83, 11.44, 17.48, 0.0, 5.24, 19.53, 63.65, 40.15, 22.75, 44.4, 33.73, 27.74, 22.41, 36.21, 32.99, 31.01, 24.72, 29.83, 37.92, 19.52, 42.22, 24.66, 11.19, 31.59, 15.24, 32.43, 17.48, 56.11, 52.62, 44.84, 49.79, 42.94, 49.05, 30.61, 43.17, 34.52, 31.5, 19.8, 17.48, 15.24, 44.4, 33.73, 42.22, 34.52, 15.24, 15.24, 42.22, 17.48, 17.48, 42.22, 24.66, 42.22, 31.5, 31.5, 31.5, 0.0, 15.24, 15.24, 15.24, 15.24, 27.74, 17.48, 17.48, 17.48, 17.48, 30.61, 17.48, 17.48, 40.15, 34.52, 17.48, 15.24, 15.24, 17.48, 33.73, 56.11, 56.11, 56.11, 31.27, 31.27, 31.27, 31.27, 31.27, 55.15, 55.15, 69.08, 42.23, 42.23, 24.71]])

Constant(name='Miles per gallon (miles/gallon)', symbol='mpg', value=9.0)

Constant(name='Dollars per gallon ($/gallon)', symbol='dpg', value=3.9)

Constant(name='Driver cost per trip ($)', symbol='dcpt', value=80)

TensorConstant(name='What cities are adjacent to a given site', symbol='s_adj', shape=[53, 97], values=['List', ['List', 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

TensorConstant(name='City populations', symbol='cpop', shape=[53, 1], values=['List', ['List', 5334], ['List', 837], ['List', 3967], ['List', 517], ['List', 560], ['List', 3565], ['List', 2160], ['List', 1102], ['List', 1791], ['List', 7117], ['List', 774], ['List', 1923], ['List', 1350], ['List', 525], ['List', 969], ['List', 1455], ['List', 7947], ['List', 676], ['List', 2177], ['List', 35579], ['List', 35999], ['List', 3046], ['List', 601], ['List', 706], ['List', 3034], ['List', 946], ['List', 4456], ['List', 966], ['List', 1204], ['List', 1067], ['List', 8397], ['List', 2198], ['List', 793], ['List', 6698], ['List', 9957], ['List', 749], ['List', 14115], ['List', 1770], ['List', 1320], ['List', 1250], ['List', 1184], ['List', 809], ['List', 536], ['List', 25571], ['List', 4065], ['List', 2222], ['List', 807], ['List', 40313], ['List', 13046], ['List', 1558], ['List', 1492], ['List', 567], ['List', 564]])

Constant(name='Total population of interest', symbol='tpop', value=272331.0)

## Variables

In [9]:
sv = TensorVariable(
  name="Sites visited",                  
  symbol="sv",
  variable_type=VariableTypeEnum.integer,
  shape=[sites.shape[0],1],
  lowerbounds=0,
  upperbounds=1,
  initial_values=0)

display(sv)

cover = TensorVariable(
    name="Coverage of cities",
    symbol="cover", 
    variable_type=VariableTypeEnum.integer,
    shape=[adjTTime.shape[0],1],
    lowerbounds=0,
    upperbounds=1,
    initial_values=0)

cover



TensorVariable(name='Sites visited', symbol='sv', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[97, 1], lowerbounds=0, upperbounds=1, initial_values=0)

TensorVariable(name='Coverage of cities', symbol='cover', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[53, 1], lowerbounds=0, upperbounds=1, initial_values=0)

## Constraints

Number of sites is $s$, number of cities is $c$

`sv` = 
$\begin{bmatrix}
x_1\\
x_2\\
⋮ \\
x_s\\
\end{bmatrix}$

- determines whether or not a given site is visited. 
- a binary vector of size $s$ 
- $s$ is the number of sites under consideration


`cover` = 
$\begin{bmatrix}
x^{'}_1\\
x^{'}_2\\
⋮ \\
x^{'}_c\\
\end{bmatrix}$

- A support variable that represents whether a city is covered by the given subset of selected sites in `sv`.
- A binary vector of size $c$
- $c$ is the number of cities under consideration

`s_adj` = 
$\begin{bmatrix}
d_{11},…,d_{1s}   \\
⋮,⋱,⋮,\\
d_{c1},…,d_{cs}   \\
\end{bmatrix}$

- A constant that represents whether or not a given site (in the columns) is adjacent to a give city (in the rows)




In [10]:
# sv = [18, 1]
# cover [28, 1]
# s_adj [28, 18]
# [28, 18] . [18, 1] = [28, 1]


g_desert = Constraint(
      name="Desert constraint",
      symbol="c",
      func="cover-s_adj@sv",
      cons_type=ConstraintTypeEnum.LTE,
      is_convex=False,
      is_linear=True,
      is_twice_differentiable=True)


g_max_site = Constraint(
      name="Keep under",
      symbol="mc",
      func="Sum(sv) - max_events",
      cons_type=ConstraintTypeEnum.LTE,
      is_convex=True,
      is_linear=True,
      is_twice_differentiable=True)



## Objectives

In [11]:

# REMEMBER! If you change the objective, you may need to change the table formatting in step02
# Total patients seen 
total_patients = Objective(
    name = "Maximize total patients served",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = "Sum(ea@sv)"                                    
)

# Underattended sites
under_attended_sites = Objective(
    name = "Minimize the number sites taht are under-attended", 
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(eua@sv)"                                
)

# Cost of events
costs = Objective(
    name = "Minimize total costs per month ($)",
    symbol = "f_3",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(sd2h@sv)*mpg_inv_times_dpg + Sum(sv)*dcpt"    
)

# cpop: [1, 28]
# cover: [28, 1]
coverage = Objective(
    name = "Maximize population with clinic access", 
    symbol = "f_4", 
    maximize = True, 
    is_twice_differentiable=True,
    func = "Sum(cover*cpop)"
)

cumul_svi = Objective(
    name = "Cumulative SVI", 
    symbol = "f_5", 
    maximize = True, 
    is_twice_differentiable=True, 
    func = "Sum(svi@sv)"
)


## Problem

In [12]:
prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        is_linear=True,
        is_convex=False,
        is_twice_differentiable=True,
        constraints=[g_desert, g_max_site],
        constants=[eua, ea, sd2h, mpg, dpg, dcpt, s_adj, city_pops, total_pop, mpg_inv_times_dpg, svi, max_events],
        variables=[sv, cover],
        objectives=[total_patients, under_attended_sites, costs, coverage, cumul_svi]
    )

with open("../../data/clinicOptProb.json", "w") as f:
    f.write(prob.model_dump_json(indent=2))

## Ideal/nadir

In [13]:

# f_1 Total patients seen 
# f_2 Underattended sites
# f_3 Cost of events
# f_4 Number of citizens covered
# f_5 max cumulative SVI 

# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(sites.loc[:,"medAtt"]))

# f_2 ideal is having no under-attended sites
# f_2 naird is having visiting all locations with under-attended sites
all_ose = int(np.sum(raw_under_attendance))


# How many miles are driven/gas costs
max_dist = sites.loc[:,"dist2home"].sum()
total_gas_cost = (max_dist / raw_mpg) * raw_dollars_per_gallon
driver_cost = sites.shape[0]*raw_driver_cost_per_trip
max_costs = float(driver_cost + total_gas_cost)

# Total number of people
total_citizens = int(cities.loc[:,["pop"]].sum())

# Maximum SVI score
max_cumul_svi = float(np.sum(raw_svi))

prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0, 
        "f_3": 0, 
        "f_4": total_citizens,
        "f_5": max_cumul_svi
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose,
        "f_3": max_costs, 
        "f_4": 0,
        "f_5": 0
        }
    )


print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")

Ideal values: {'f_1': 404, 'f_2': 0, 'f_3': 0, 'f_4': 272331, 'f_5': 58.723883328}
Nadir values: {'f_1': 0, 'f_2': 57, 'f_3': 8988.881333333333, 'f_4': 0, 'f_5': 0}


## RPM solver


In [14]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions
from itertools import product
from desdeo.tools import PyomoBonminSolver, PyomoCBCSolver, PyomoGurobiSolver, PyomoIpoptSolver
from desdeo.tools import NevergradGenericSolver

ref_resolution = 3

f_1_ref = np.linspace(prob.get_ideal_point()["f_1"], prob.get_nadir_point()["f_1"],ref_resolution).tolist()
f_2_ref = np.linspace(prob.get_ideal_point()["f_2"], prob.get_nadir_point()["f_2"],ref_resolution).tolist()
f_3_ref = np.linspace(prob.get_ideal_point()["f_3"], prob.get_nadir_point()["f_3"],ref_resolution).tolist()
f_4_ref = np.linspace(prob.get_ideal_point()["f_4"], prob.get_nadir_point()["f_4"],ref_resolution).tolist()
f_5_ref = np.linspace(prob.get_ideal_point()["f_5"], prob.get_nadir_point()["f_5"],ref_resolution).tolist()
#f_1_ref = [200]
#f_2_ref = [10]
#f_3_ref = [1000]
#f_4_ref = [0.5]

pf_samples_raw = []
i = 0 
for ref in product(f_1_ref, f_2_ref, f_3_ref, f_4_ref, f_5_ref):
    reference_point = {"f_1": ref[0], "f_2": ref[1], "f_3": ref[2], "f_4": ref[3], "f_5": ref[4]}

    print(f"Calculating for ref #{i} {reference_point}")
    try: 
        res = rpm_solve_solutions(prob, reference_point=reference_point, solver=PyomoGurobiSolver)
        pf_samples_raw.append(res)
    except ValueError:
        print("Error running for that ref.")
    i += 1


Calculating for ref #0 {'f_1': 404.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 272331.0, 'f_5': 58.723883328}
SetProduct.subsets() to get the operator arguments.  (deprecated in 5.7)
(called from C:\Users\i-
kropp\AppData\Local\DESDEO_fc70255\desdeo\problem\json_parser.py:455)
SetProduct.subsets() to get the operator arguments.  (deprecated in 5.7)
(called from C:\Users\i-
kropp\AppData\Local\DESDEO_fc70255\desdeo\problem\json_parser.py:456)
SetProduct.subsets() to get the operator arguments.  (deprecated in 5.7)
(called from C:\Users\i-
kropp\AppData\Local\DESDEO_fc70255\desdeo\problem\json_parser.py:461)
SetProduct.subsets() to get the operator arguments.  (deprecated in 5.7)
(called from C:\Users\i-
kropp\AppData\Local\DESDEO_fc70255\desdeo\problem\json_parser.py:469)
SetProduct.subsets() to get the operator arguments.  (deprecated in 5.7)
(called from C:\Users\i-
kropp\AppData\Local\DESDEO_fc70255\desdeo\problem\json_parser.py:470)
SetProduct.subsets() to get the operator arguments.  (deprec

## Send relevant data to a pkl file

In [16]:
import pickle
output = open(f'{data_home}/pf_test.pkl', 'wb')
pickle.dump({"pf": pf_samples_raw,
             "prob" : prob,
             "cities": cities,
             "sites": sites,
             "cities_adj2sites": cities_adj2sites, 
             "total_pop": total_citizens
             }, output)
